# Recherche de facteurs financiers — STOXX Europe 600

Ce notebook exécute une recherche Top/Worst propre aux banques, aux assurances et aux services financiers du STOXX Europe 600. Les scores et les benchmarks sont calculés à l'intérieur de chaque segment financier.

Principes de construction :

- exclusion des multiples EV/EBITDA, P/FCF et des ratios de dette industrielle ;
- modèle bancaire fondé sur rentabilité, capital, qualité du crédit et tangible book ;
- ratio combiné limité au module des assureurs dommages ;
- révisions, momentum, dividende et faible volatilité utilisés comme confirmations ;
- sélection finale fondée sur la surperformance totale et la stabilité sur six sous-périodes complètes.

Le dossier `exports/financial_sector_research/stoxx600` contient les CSV détaillés et un fichier `shareable_results.txt` prêt à copier pour une analyse ultérieure.

In [ ]:
from pathlib import Path
import importlib
import sys
import pandas as pd

PLUGIN_DIR = Path(r"C:\dev\factor_backtest")
EXPORTS_DIR = PLUGIN_DIR / "exports"
if str(EXPORTS_DIR) not in sys.path:
    sys.path.insert(0, str(EXPORTS_DIR))

import financial_factor_pipeline as financial_pipeline
importlib.reload(financial_pipeline)

print("Pipeline financier chargé.")

## Paramètres de l'expérience

`N_JOBS=1` donne l'exécution la plus lisible dans Jupyter sous Windows. Une valeur supérieure accélère le calcul mais laisse apparaître davantage de messages du moteur. Le percentile de 20 % conserve un nombre raisonnable de titres dans les sous-secteurs financiers.

In [ ]:
MARKET_KEY = "stoxx600"
START_DATE = "2009-02-01"
PERCENTILE = 0.20
N_JOBS = 1
MINIMUM_COVERAGE = 0.60
MINIMUM_MEDIAN_NAMES = 10
OUTPUT_ROOT = EXPORTS_DIR / "financial_sector_research"

display(financial_pipeline.candidate_table())

## Exécution complète

La cellule suivante recharge les données, applique les filtres de couverture, teste les dimensions autorisées, exécute les composites et exporte les résultats. Le `noyau persistant` des assurances est explicitement une hypothèse dérivée après screening croisé : son backtest reste in-sample et ne constitue pas une validation future. La période depuis 2026 est imprimée comme information tactique, mais elle n'entre pas dans la porte de persistance.

In [ ]:
result = financial_pipeline.run_market_experiment(
    market_key=MARKET_KEY,
    output_root=OUTPUT_ROOT,
    start_date=START_DATE,
    percentile=PERCENTILE,
    n_jobs=N_JOBS,
    minimum_coverage=MINIMUM_COVERAGE,
    minimum_median_names=MINIMUM_MEDIAN_NAMES,
)

## Contrôle de la couverture et résultats qui battent le benchmark

La première table montre les variables retenues ou rejetées avant le backtest. La deuxième ne montre que les tests dont l'active CAGR totale est positive ; une place dans cette table ne suffit donc pas à obtenir la porte stricte ou la porte de persistance.

In [ ]:
coverage_columns = [
    "segment", "variable", "family", "coverage",
    "median_valid_names", "selected", "selection_reason",
]
display(result["coverage"][coverage_columns])

result_columns = [
    "segment", "test_name", "test_type", "family",
    "persistent_gate", "total_gate", "active_cagr",
    "top_worst_cagr", "top_information_ratio", "robust_score",
    "positive_periods", "completed_periods", "worst_active_cagr",
    "current_active_cagr", "current_information_ratio",
]
outperformers = result["ranked"].loc[result["ranked"]["active_cagr"].gt(0)]
display(outperformers[result_columns].groupby("segment", group_keys=False).head(12))

## Bloc compact à partager

Copiez intégralement la sortie de cette cellule pour demander une nouvelle analyse. Elle exclut déjà les tests qui ne battent pas le benchmark sur la période totale et conserve les indicateurs de stabilité.

In [ ]:
financial_pipeline.print_shareable_results(result["output_dir"])